In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,random_split
from torchvision import datasets
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
dataset=datasets.MNIST(root='./mnist',train=True,download=True,transform=transforms.ToTensor())
train_set,val_set=random_split(dataset,[50000,10000])
test_set=datasets.MNIST(root='./mnist',train=False,download=True,transform=transforms.ToTensor())

train_loader=DataLoader(train_set,batch_size=500,shuffle=True)
val_loader=DataLoader(val_set,batch_size=500,shuffle=True)
test_loader=DataLoader(test_set,batch_size=500,shuffle=True)

In [2]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,20,(5,5)) #(B,20,24,24)
        self.pool1=nn.MaxPool2d(2,2) #(B,20,12,12)
        self.conv2=nn.Conv2d(20,40,(5,5)) #(B,40,8,8)
        self.pool2=nn.MaxPool2d(2,2) #(B,40,4,4)
        self.fc1=nn.Linear(640,120) #120自己定义的,40*4*4
        self.fc2=nn.Linear(120,10) #10分类的问题

    def forward(self,x):  #x :(B,1,28,28)
        B=x.shape[0]
        x=F.relu(self.conv1(x))
        x=self.pool1(x)
        x=F.relu(self.conv2(x))
        x=self.pool2(x)
        x=F.relu(self.fc1(x.view(B,-1))) #Linear要求输入形状是（B,特征数）
        x=self.fc2(x)
        return x

model=CNN()

In [3]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.001)

epochs=10

for epoch in range(epochs):
    model.train()
    loss_train=0

    for images,labels in train_loader:
        preds=model(images)
        loss=criterion(preds,labels)
        loss_train+=loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    loss_val=0

    with torch.no_grad():
        for images,labels in val_loader:
            preds=model(images)
            loss=criterion(preds,labels)
            loss_val+=loss.item()

    print(f"epoch:{epoch+1} loss_train={loss_train/len(train_loader)},loss_val={loss_val/len(val_loader)}")







epoch:1 loss_train=0.6287881109118462,loss_val=0.1998064249753952
epoch:2 loss_train=0.1403755886852741,loss_val=0.11814879849553109
epoch:3 loss_train=0.09076137337833642,loss_val=0.09578818306326867
epoch:4 loss_train=0.07120903674513102,loss_val=0.06977830976247787
epoch:5 loss_train=0.058945798166096214,loss_val=0.07167089320719242
epoch:6 loss_train=0.052632474303245545,loss_val=0.06638760101050138
epoch:7 loss_train=0.0472931364364922,loss_val=0.06159945297986269
epoch:8 loss_train=0.04399427173659205,loss_val=0.05409886250272393
epoch:9 loss_train=0.04050459056161344,loss_val=0.0495771006681025
epoch:10 loss_train=0.039341843696311114,loss_val=0.0499242234043777


In [4]:
correct=0
total=0

with torch.no_grad():
    for images,labels in test_loader: #每次循环会取出一个batch,labels.shape=(500,)
        preds=model(images)
        _,predicted = torch.max(preds,1)

        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()

    print(f"准确率: {correct/total*100}%")



准确率: 98.69%
